Инициализация (запускать после каждого перезапуска среды)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
import numpy as np
import pandas as pd
import tensorflow as tf

DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/....' # ВСТАВИТЬ НУЖНЫЙ АДРЕС
CONFIG   = f'{DATA_DIR}/config.json'

if DATA_DIR not in sys.path:
    sys.path.insert(0, DATA_DIR)

print(f'DATA_DIR: {DATA_DIR}')
print(f'TensorFlow: {tf.__version__}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_DIR: /content/drive/MyDrive/Colab Notebooks/WORK/DATA
TensorFlow: 2.20.0


Батч-обработка всех пациентов

In [ ]:
# Сброс кэша и перезапуск
import sys
for mod in ['batch_pwv', 'pwv_single', 'pwv_cnn']:
    if mod in sys.modules:
        del sys.modules[mod]

from batch_pwv import run_batch, plot_patient_comparison

df_signals, df_patients, df_features, df_good = run_batch(
    config_path = CONFIG,
    data_dir    = DATA_DIR,
    plot        = False,
)
display(df_good)


############################################################
  ПАЦИЕНТ P01   dist=0.57 м   tkeo=0.8
  Файлов: 14
############################################################

Файл: /content/drive/MyDrive/Colab Notebooks/WORK/DATA/data4ch_0_1.csv
Строк: 24997, столбцов: 5
Длина записи: 51.2 с
Артефактов (маска): 23.8%
R-пиков найдено: 79
Foot груди: 52,  пар с рукой: 52

Результаты:
  Валидных пар: 52
  PTT медиана: 64.5 мс  std: 72.0 мс
  PWV медиана: 8.84 м/с  std: 9.98 м/с
  HR: 97.5 уд/мин  RMSSD: 154.5 мс  SDNN: 162.9 мс

Файл: /content/drive/MyDrive/Colab Notebooks/WORK/DATA/data4ch_0_2.csv
Строк: 31749, столбцов: 5
Длина записи: 65.0 с
Артефактов (маска): 26.3%
R-пиков найдено: 99
Foot груди: 67,  пар с рукой: 67

Результаты:
  Валидных пар: 67
  PTT медиана: 92.2 мс  std: 81.1 мс
  PWV медиана: 6.18 м/с  std: 11.28 м/с
  HR: 97.5 уд/мин  RMSSD: 152.6 мс  SDNN: 150.3 мс

Файл: /content/drive/MyDrive/Colab Notebooks/WORK/DATA/data4ch_0_3.csv
Строк: 32140, столбцов: 5
Длина записи

,patient_id,signal_idx,file,quality_ok,duration_s,artifact_pct,n_rpeaks,n_valid,valid_ratio,ptt_median_ms,...,ptt_iqr_ms,pwv_median_ms,pwv_std,pwv_p10,pwv_p90,mean_hr,sdnn_ms,rmssd_ms,pnn50,rr_cv
2,P01,3,/content/drive/MyDrive/Colab Notebooks/WORK/DA...,True,65.8,26.7,101,67,0.663,153.6,...,177.152454,3.71,10.51,2.869274,27.831960,98.3,182.4,206.7,52.5,0.3102
3,P01,4,/content/drive/MyDrive/Colab Notebooks/WORK/DA...,True,42.7,22.2,67,39,0.582,182.3,...,32.768084,3.13,1.34,2.869274,5.703658,105.3,129.0,151.3,31.6,0.2843
4,P01,5,/content/drive/MyDrive/Colab Notebooks/WORK/DA...,True,58.9,27.0,96,66,0.688,142.3,...,151.040387,4.00,10.06,2.869274,27.831960,106.9,113.8,120.9,34.9,0.2773
5,P01,6,/content/drive/MyDrive/Colab Notebooks/WORK/DA...,True,55.7,30.0,86,57,0.663,106.5,...,34.816089,5.35,6.04,3.337280,9.513470,97.8,142.8,130.1,14.8,0.2488
6,P01,7,/content/drive/MyDrive/Colab Notebooks/WORK/DA...,True,67.5,20.6,111,65,0.586,98.3,...,18.432047,5.80,1.39,4.475709,6.825457,106.4,131.6,146.5,42.9,0.2779
8,P01,9,/content/drive/MyDrive/Colab Notebooks/WORK/DA...,True,73.9,18.7,118,90,0.763,106.5,...,79.872204,5.35,7.34,2.869274,23.404148,102.6,113.5,118.5,31.7,0.2382
9,P01,10,/content/drive/MyDrive/Colab Notebooks/WORK/DA...,True,51.2,19.5,81,62,0.765,165.9,...,102.400262,3.44,8.14,2.869274,27.831960,99.1,146.7,130.9,29.8,0.2692
11,P01,12,/content/drive/MyDrive/Colab Notebooks/WORK/DA...,True,53.2,21.2,75,50,0.667,138.2,...,160.256410,4.13,11.12,2.869274,27.831960,89.2,111.5,105.2,33.3,0.2282
12,P01,13,/content/drive/MyDrive/Colab Notebooks/WORK/DA...,True,83.7,21.1,103,77,0.748,116.7,...,71.680184,4.88,8.53,2.869274,27.831960,86.1,188.3,193.8,32.0,0.3022
21,P03,3,/content/drive/MyDrive/Colab Notebooks/WORK/DA...,True,89.4,34.5,172,102,0.593,102.4,...,34.816089,3.71,4.89,2.416119,18.385961,125.6,103.8,96.2,10.0,0.3304


Дашборд сравнения пациентов

In [ ]:
patient_meta = {
    'P01': {'age': 26, 'notes': 'аритмия, лишний вес'},
    'P02': {'age': 40, 'notes': 'норма'},
    'P03': {'age': 55, 'notes': 'норма, жен.'},
}

plot_patient_comparison(df_features, patient_meta)

Детальный анализ одного сигнала

In [ ]:
from pwv_single import process_file

result = process_file(
    filepath      = f'{DATA_DIR}/data4ch_0_7.csv',
    distance_m    = 0.57,
    tkeo_factor   = 0.8,
    plot          = True,
    verbose       = True,
)

print(f'СРПВ:  {result["pwv_ms"]:.2f} м/с')
print(f'PTT:   {result["ptt_ms"]:.1f} мс')
print(f'ЧСС:   {result["mean_hr"]:.1f} уд/мин')
print(f'SDNN:  {result["sdnn_ms"]:.1f} мс')
print(f'RMSSD: {result["rmssd_ms"]:.1f} мс')
print(f'pNN50: {result["pnn50"]:.1f} %')

Output hidden; open in https://colab.research.google.com to view.

Сборка датасета для CNN

In [ ]:
from pwv_cnn import build_dataset
from collections import Counter
import numpy as np

X, y, meta = build_dataset(CONFIG, DATA_DIR)

valid_mask = (y > 25) & (y <= 250)
X_clean    = X[valid_mask]
y_clean    = y[valid_mask]
meta_clean = [meta[i] for i, v in enumerate(valid_mask) if v]

print(f'Битов: {len(y_clean)}')
print(pd.DataFrame(meta_clean)['patient_id'].value_counts())

# Веса для балансировки пациентов
counts = Counter(m['patient_id'] for m in meta_clean)
total  = len(meta_clean)
sample_weights = np.array([
    total / (len(counts) * counts[m['patient_id']])
    for m in meta_clean
])
print('\nВеса по пациентам:')
for pid, cnt in counts.items():
    w = total / (len(counts) * cnt)
    print(f'  {pid}: {cnt} битов → вес {w:.3f}')

quality_csv не задан — берём все сигналы
  Извлечение битов: data4ch_0_1.csv ... 52 битов
  Извлечение битов: data4ch_0_2.csv ... 67 битов
  Извлечение битов: data4ch_0_3.csv ... 67 битов
  Извлечение битов: data4ch_0_4.csv ... 39 битов
  Извлечение битов: data4ch_0_5.csv ... 65 битов
  Извлечение битов: data4ch_0_6.csv ... 57 битов
  Извлечение битов: data4ch_0_7.csv ... 65 битов
  Извлечение битов: data4ch_0_8.csv ... 65 битов
  Извлечение битов: data4ch_0_9.csv ... 90 битов
  Извлечение битов: data4ch_0_10.csv ... 62 битов
  Извлечение битов: data4ch_0_11.csv ... 22 битов
  Извлечение битов: data4ch_0_12.csv ... 50 битов
  Извлечение битов: data4ch_0_13.csv ... 77 битов
  Извлечение битов: data4ch_0_14.csv ... 0 битов
  Извлечение битов: data4ch_1_1.csv ... 51 битов
  Извлечение битов: data4ch_1_2.csv ... 80 битов
  Извлечение битов: data4ch_1_3.csv ... 141 битов
  Извлечение битов: data4ch_1_4.csv ... 180 битов
  Извлечение битов: data4ch_1_5.csv ... 50 битов
  Извлечение битов: da

Обучение CNN

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from pwv_cnn import build_cnn

K.clear_session()

# Split
train_idx, test_idx = train_test_split(
    list(range(len(y_clean))),
    test_size    = 0.2,
    random_state = 42,
    stratify     = [m['patient_id'] for m in meta_clean],
)

X_train, X_test = X_clean[train_idx], X_clean[test_idx]
y_train, y_test = y_clean[train_idx], y_clean[test_idx]
meta_test       = [meta_clean[i] for i in test_idx]

# Веса только для обучающей выборки
w_train = sample_weights[train_idx]

# Модель
model = build_cnn(input_shape=(X_clean.shape[1], 3))
model.compile(optimizer=Adam(learning_rate=1e-3),
              loss='huber', metrics=['mae'])

history = model.fit(
    X_train, y_train,
    sample_weight    = w_train,
    validation_split = 0.15,
    epochs           = 60,
    batch_size       = 32,
    callbacks        = [
        EarlyStopping(patience=12, restore_best_weights=True,
                      monitor='val_mae'),
        ReduceLROnPlateau(factor=0.5, patience=6,
                          min_lr=1e-5, monitor='val_mae'),
        ModelCheckpoint(f'{DATA_DIR}/pwv_cnn_v2.h5',
                        save_best_only=True, monitor='val_mae'),
    ],
    verbose = 1,
)

Epoch 1/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - loss: 126.4552 - mae: 124.3303

28/28 ━━━━━━━━━━━━━━━━━━━━ 9s 198ms/step - loss: 113.6610 - mae: 114.7388 - val_loss: 125.4321 - val_mae: 122.9289 - learning_rate: 0.0010
Epoch 2/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - loss: 54.4517 - mae: 56.5362

28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 97ms/step - loss: 51.9446 - mae: 51.6080 - val_loss: 69.8866 - val_mae: 67.7716 - learning_rate: 0.0010
Epoch 3/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - loss: 40.2057 - mae: 40.7483

28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 96ms/step - loss: 42.4571 - mae: 42.7223 - val_loss: 55.2868 - val_mae: 51.9695 - learning_rate: 0.0010
Epoch 4/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - loss: 39.8156 - mae: 41.9384

28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 96ms/step - loss: 40.8970 - mae: 41.9591 - val_loss: 43.3791 - val_mae: 40.0938 - learning_rate: 0.0010
Epoch 5/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - loss: 40.8172 - mae: 41.5332

...

28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 96ms/step - loss: 28.0687 - mae: 28.9187 - val_loss: 31.6818 - val_mae: 29.1267 - learning_rate: 0.0010
Epoch 30/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - loss: 27.0179 - mae: 29.6361

28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 96ms/step - loss: 26.6467 - mae: 28.2435 - val_loss: 29.9387 - val_mae: 27.9985 - learning_rate: 0.0010
Epoch 31/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 165ms/step - loss: 28.2964 - mae: 29.1040 - val_loss: 37.8519 - val_mae: 35.9005 - learning_rate: 0.0010
Epoch 32/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - loss: 26.2875 - mae: 27.4249

28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 96ms/step - loss: 27.8113 - mae: 28.1343 - val_loss: 30.9723 - val_mae: 27.9336 - learning_rate: 0.0010
Epoch 33/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 92ms/step - loss: 26.2054 - mae: 27.6098 - val_loss: 39.5920 - val_mae: 36.8982 - learning_rate: 0.0010
Epoch 34/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 6s 107ms/step - loss: 27.5380 - mae: 28.6112 - val_loss: 32.6909 - val_mae: 31.2700 - learning_rate: 0.0010
Epoch 35/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 162ms/step - loss: 27.9211 - mae: 28.9229 - val_loss: 30.7967 - val_mae: 29.5417 - learning_rate: 0.0010
Epoch 36/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - loss: 26.9347 - mae: 28.4494

28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 111ms/step - loss: 27.3931 - mae: 28.2622 - val_loss: 28.0647 - val_mae: 26.4041 - learning_rate: 0.0010
Epoch 37/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 93ms/step - loss: 26.8822 - mae: 27.4985 - val_loss: 29.7251 - val_mae: 28.2177 - learning_rate: 0.0010
Epoch 38/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 92ms/step - loss: 25.4444 - mae: 27.4341 - val_loss: 32.7470 - val_mae: 30.7023 - learning_rate: 0.0010
Epoch 39/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 122ms/step - loss: 25.9801 - mae: 26.7738 - val_loss: 34.9675 - val_mae: 31.0632 - learning_rate: 0.0010
Epoch 40/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 96ms/step - loss: 25.9719 - mae: 27.3840 - val_loss: 32.5152 - val_mae: 28.6018 - learning_rate: 0.0010
Epoch 41/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 94ms/step - loss: 25.9486 - mae: 27.3817 - val_loss: 31.7858 - val_mae: 29.2606 - learning_rate: 0.0010
Epoch 42/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 93ms/step - loss: 25.5655 - mae: 27.3200 - val_loss: 32.2878 - val_mae: 30.0942 - learning_r

28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 121ms/step - loss: 22.3134 - mae: 24.1072 - val_loss: 27.1631 - val_mae: 25.2676 - learning_rate: 5.0000e-04
Epoch 44/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 101ms/step - loss: 22.0342 - mae: 23.6679 - val_loss: 33.1290 - val_mae: 31.4727 - learning_rate: 5.0000e-04
Epoch 45/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 92ms/step - loss: 22.0941 - mae: 24.0249 - val_loss: 32.3066 - val_mae: 28.2870 - learning_rate: 5.0000e-04
Epoch 46/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 93ms/step - loss: 21.8591 - mae: 23.4054 - val_loss: 28.1448 - val_mae: 25.6304 - learning_rate: 5.0000e-04
Epoch 47/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 6s 111ms/step - loss: 21.4376 - mae: 22.7306 - val_loss: 32.8859 - val_mae: 28.4881 - learning_rate: 5.0000e-04
Epoch 48/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 92ms/step - loss: 20.7816 - mae: 22.8343 - val_loss: 29.8803 - val_mae: 27.2683 - learning_rate: 5.0000e-04
Epoch 49/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 93ms/step - loss: 21.4640 - mae: 23.3369 - val_loss: 31.6301 - val_

Оценка CNN

In [ ]:
from pwv_cnn import evaluate_cnn, plot_training_history

plot_training_history(history)
metrics = evaluate_cnn(model, X_test, y_test, meta_test)

baseline = float(np.mean(np.abs(np.median(y_train) - y_test)))
print(f'Baseline MAE: {baseline:.1f} мс')
print(f'CNN MAE:      {metrics["mae"]:.1f} мс')
print(f'Улучшение:    {(1 - metrics["mae"]/baseline)*100:.0f}%')


Метрики на тесте:
  MAE  = 28.77 мс
  RMSE = 38.91 мс
  R²   = 0.430


Baseline MAE: 45.5 мс
CNN MAE:      28.8 мс
Улучшение:    37%


Загрузка сохранённой модели (при необходимости, вместо обучения)

In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model(f'{DATA_DIR}/pwv_cnn_v2.h5')
print('Модель загружена:', model.name)
print('Входная форма:', model.input_shape)

In [ ]:
# Переобучить без весов
K.clear_session()
model = build_cnn(input_shape=(X_clean.shape[1], 3))
model.compile(optimizer=Adam(learning_rate=1e-3),
              loss='huber', metrics=['mae'])

history = model.fit(
    X_train, y_train,          # без sample_weight
    validation_split = 0.15,
    epochs=60, batch_size=32,
    callbacks=[
        EarlyStopping(patience=12, restore_best_weights=True, monitor='val_mae'),
        ReduceLROnPlateau(factor=0.5, patience=6, min_lr=1e-5, monitor='val_mae'),
        ModelCheckpoint(f'{DATA_DIR}/pwv_cnn_v2.h5',
                        save_best_only=True, monitor='val_mae'),
    ], verbose=1,
)

Epoch 1/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step - loss: 119.6087 - mae: 120.0950

28/28 ━━━━━━━━━━━━━━━━━━━━ 9s 174ms/step - loss: 102.0976 - mae: 102.5828 - val_loss: 107.8549 - val_mae: 108.3405 - learning_rate: 0.0010
Epoch 2/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - loss: 44.5485 - mae: 45.0311

28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 99ms/step - loss: 44.8908 - mae: 45.3735 - val_loss: 58.7925 - val_mae: 59.2735 - learning_rate: 0.0010
Epoch 3/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 93ms/step - loss: 40.5714 - mae: 41.0523 - val_loss: 71.3044 - val_mae: 71.7884 - learning_rate: 0.0010
Epoch 4/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step - loss: 36.2415 - mae: 36.7231

28/28 ━━━━━━━━━━━━━━━━━━━━ 6s 132ms/step - loss: 38.6156 - mae: 39.0978 - val_loss: 46.6124 - val_mae: 47.0954 - learning_rate: 0.0010
Epoch 5/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - loss: 37.4337 - mae: 37.9142

28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 97ms/step - loss: 35.9511 - mae: 36.4324 - val_loss: 46.6105 - val_mae: 47.0917 - learning_rate: 0.0010
Epoch 6/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - loss: 33.0485 - mae: 33.5284

...

28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 96ms/step - loss: 29.6130 - mae: 30.0888 - val_loss: 27.9200 - val_mae: 28.3954 - learning_rate: 0.0010
Epoch 17/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 92ms/step - loss: 27.9030 - mae: 28.3757 - val_loss: 28.4270 - val_mae: 28.9021 - learning_rate: 0.0010
Epoch 18/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 141ms/step - loss: 29.2477 - mae: 29.7241 - val_loss: 40.1150 - val_mae: 40.5925 - learning_rate: 0.0010
Epoch 19/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - loss: 28.2227 - mae: 28.6991

28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 131ms/step - loss: 28.3229 - mae: 28.7997 - val_loss: 26.2252 - val_mae: 26.7047 - learning_rate: 0.0010
Epoch 20/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 91ms/step - loss: 28.0762 - mae: 28.5501 - val_loss: 26.4469 - val_mae: 26.9219 - learning_rate: 0.0010
Epoch 21/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 93ms/step - loss: 28.3618 - mae: 28.8358 - val_loss: 34.7958 - val_mae: 35.2746 - learning_rate: 0.0010
Epoch 22/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 108ms/step - loss: 26.6928 - mae: 27.1687 - val_loss: 31.3347 - val_mae: 31.8070 - learning_rate: 0.0010
Epoch 23/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 159ms/step - loss: 26.6027 - mae: 27.0781 - val_loss: 26.2989 - val_mae: 26.7711 - learning_rate: 0.0010
Epoch 24/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - loss: 26.4974 - mae: 26.9698

28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 111ms/step - loss: 27.3714 - mae: 27.8436 - val_loss: 23.3065 - val_mae: 23.7738 - learning_rate: 0.0010
Epoch 25/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 92ms/step - loss: 26.0727 - mae: 26.5461 - val_loss: 25.5967 - val_mae: 26.0640 - learning_rate: 0.0010
Epoch 26/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 93ms/step - loss: 25.9320 - mae: 26.4047 - val_loss: 25.6588 - val_mae: 26.1309 - learning_rate: 0.0010
Epoch 27/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - loss: 26.7773 - mae: 27.2495

28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 126ms/step - loss: 27.0462 - mae: 27.5185 - val_loss: 23.2201 - val_mae: 23.6894 - learning_rate: 0.0010
Epoch 28/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 143ms/step - loss: 25.6382 - mae: 26.1111 - val_loss: 25.0789 - val_mae: 25.5553 - learning_rate: 0.0010
Epoch 29/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 92ms/step - loss: 25.5242 - mae: 25.9955 - val_loss: 32.1249 - val_mae: 32.5917 - learning_rate: 0.0010
Epoch 30/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 92ms/step - loss: 24.9661 - mae: 25.4352 - val_loss: 24.2397 - val_mae: 24.7078 - learning_rate: 0.0010
Epoch 31/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 92ms/step - loss: 25.9683 - mae: 26.4381 - val_loss: 26.3604 - val_mae: 26.8350 - learning_rate: 0.0010
Epoch 32/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 7s 153ms/step - loss: 24.1755 - mae: 24.6450 - val_loss: 28.1447 - val_mae: 28.6170 - learning_rate: 0.0010
Epoch 33/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 94ms/step - loss: 22.8467 - mae: 23.3153 - val_loss: 32.8290 - val_mae: 33.3000 - learning_

28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 114ms/step - loss: 22.8279 - mae: 23.2966 - val_loss: 22.4646 - val_mae: 22.9386 - learning_rate: 5.0000e-04
Epoch 35/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 92ms/step - loss: 22.4427 - mae: 22.9120 - val_loss: 25.0920 - val_mae: 25.5665 - learning_rate: 5.0000e-04
Epoch 36/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 134ms/step - loss: 21.0213 - mae: 21.4889 - val_loss: 30.9157 - val_mae: 31.3887 - learning_rate: 5.0000e-04
Epoch 37/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 136ms/step - loss: 20.0192 - mae: 20.4864 - val_loss: 24.6336 - val_mae: 25.1075 - learning_rate: 5.0000e-04
Epoch 38/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 94ms/step - loss: 20.9218 - mae: 21.3905 - val_loss: 28.5134 - val_mae: 28.9859 - learning_rate: 5.0000e-04
Epoch 39/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 93ms/step - loss: 20.8931 - mae: 21.3624 - val_loss: 26.3066 - val_mae: 26.7802 - learning_rate: 5.0000e-04
Epoch 40/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 6s 129ms/step - loss: 21.1780 - mae: 21.6446 - val_loss: 29.1453 - val

In [ ]:
from pwv_cnn import evaluate_cnn, plot_training_history

plot_training_history(history)
metrics = evaluate_cnn(model, X_test, y_test, meta_test)

baseline = float(np.mean(np.abs(np.median(y_train) - y_test)))
print(f'Baseline MAE: {baseline:.1f} мс')
print(f'CNN MAE:      {metrics["mae"]:.1f} мс')
print(f'Улучшение:    {(1 - metrics["mae"]/baseline)*100:.0f}%')


Метрики на тесте:
  MAE  = 25.38 мс
  RMSE = 35.33 мс
  R²   = 0.530


Baseline MAE: 45.5 мс
CNN MAE:      25.4 мс
Улучшение:    44%


Инференс по одному сигналу на пациента

In [ ]:
p02_test = [meta_clean[i] for i in test_idx
            if meta_clean[i]['patient_id'] == 'P02']
print(f'P02 в тесте: {len(p02_test)} битов')

p02_train = [meta_clean[i] for i in train_idx
             if meta_clean[i]['patient_id'] == 'P02']
print(f'P02 в обучении: {len(p02_train)} битов')

P02 в тесте: 30 битов
P02 в обучении: 121 битов


In [ ]:
from pwv_cnn import run_inference

demo_signals = [
    dict(filepath=f'{DATA_DIR}/data4ch_0_7.csv',
         distance_m=0.57, tkeo_factor=0.8,
         patient_id='P01 (26л, аритмия)'),
    dict(filepath=f'{DATA_DIR}/data4ch_1_3.csv',
         distance_m=0.40, tkeo_factor=0.6,
         patient_id='P02 (40л, норма)'),
    dict(filepath=f'{DATA_DIR}/data4ch_4_3.csv',
         distance_m=0.38, tkeo_factor=0.4,
         patient_id='P03 (55л, норма)'),
]

results = []
for s in demo_signals:
    try:
        r = run_inference(model=model, show_plot=True, **s)
        if r: results.append(r)
    except Exception as e:
        print(f'[!] {s["patient_id"]}: {e}')

display(pd.DataFrame(results)[
    ['patient_id','algo_ptt_ms','cnn_ptt_ms',
     'agreement_ms','algo_pwv_ms','cnn_pwv_ms']])


Инференс: data4ch_0_7.csv  [P01 (26л, аритмия)]
  Алгоритм:  PTT=98.3 мс  PWV=5.80 м/с  (n=65)
  CNN:       PTT=101.0 мс  PWV=5.64 м/с  (n=81)
  Расхождение алгоритм↔CNN: 2.7 мс



Инференс: data4ch_1_3.csv  [P02 (40л, норма)]
  Алгоритм:  PTT=57.3 мс  PWV=6.98 м/с  (n=141)
  CNN:       PTT=159.5 мс  PWV=2.51 м/с  (n=139)
  Расхождение алгоритм↔CNN: 102.2 мс



Инференс: data4ch_4_3.csv  [P03 (55л, норма)]
  Алгоритм:  PTT=102.4 мс  PWV=3.71 м/с  (n=102)
  CNN:       PTT=102.4 мс  PWV=3.71 м/с  (n=96)
  Расхождение алгоритм↔CNN: 0.0 мс


,patient_id,algo_ptt_ms,cnn_ptt_ms,agreement_ms,algo_pwv_ms,cnn_pwv_ms
0,"P01 (26л, аритмия)",98.3,101.0,2.7,5.80,5.64
1,"P02 (40л, норма)",57.3,159.5,102.2,6.98,2.51
2,"P03 (55л, норма)",102.4,102.4,0.0,3.71,3.71


Уравнение Моэнса-Кортевега в линейной аппроксимации

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    'pwv_bp_calibration',
    f'{DATA_DIR}/pwv_bp_calibration.py',
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

run_calibration = mod.run_calibration
estimate_bp     = mod.estimate_bp

# Парные измерения для P01: СРПВ (м/с) + АД манжетой (мм рт.ст.)
pwv_values = [8.70, 5.25, 3.90, 3.20, 4.22, 5.46, 4.97]
sbp_values = [144,  133,  128,  126,  130,  134,  132]
dbp_values = [ 94,   84,   80,   77,   81,   84,   83]

calib = run_calibration(
    pwv_values = pwv_values,
    sbp_values = sbp_values,
    dbp_values = dbp_values,
    patient_id = 'P01',
    show_plot  = True,
)


Калибровка СРПВ → АД  [P01]
Точек измерений: 7
  СД (SBP): BP = 3.27×СРПВ + 115.74  R²=0.997  MAE=0.3 мм рт.ст.
  ДД (DBP): BP = 3.00×СРПВ + 67.98  R²=0.995  MAE=0.3 мм рт.ст.
  MAP: BP = 3.09×СРПВ + 83.90  R²=0.998  MAE=0.2 мм рт.ст.

  Корреляция Пирсона:
    СРПВ ↔ СД:  r = 0.998
    СРПВ ↔ ДД:  r = 0.998
    СРПВ ↔ MAP: r = 0.999


Оценка АД по новым записям

In [ ]:
# По диапазону СРПВ
df_bp = estimate_bp(calib, pwv_new=[4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 8.0])
display(df_bp)

# По всем записям P01
pwv_p01 = list(df_signals[df_signals['patient_id'] == 'P01']['pwv_ms'])
df_bp_p01 = estimate_bp(calib, pwv_new=pwv_p01)
df_bp_p01['signal_idx'] = list(
    df_signals[df_signals['patient_id'] == 'P01']['signal_idx'])
display(df_bp_p01[['signal_idx', 'pwv_ms', 'bp_str', 'map_est', 'in_range']])


Оценка АД по СРПВ [P01]:
 pwv_ms bp_str  map_est  in_range
    4.0 129/80     96.3      True
    4.5 130/81     97.8      True
    5.0 132/83     99.4      True
    5.5 134/84    100.9      True
    6.0 135/86    102.4      True
    6.5 137/87    104.0      True
    7.0 139/89    105.5      True
    8.0 142/92    108.6      True


,pwv_ms,sbp_est,dbp_est,map_est,bp_str,in_range
0,4.0,128.8,80.0,96.3,129/80,True
1,4.5,130.5,81.5,97.8,130/81,True
2,5.0,132.1,83.0,99.4,132/83,True
3,5.5,133.7,84.5,100.9,134/84,True
4,6.0,135.4,86.0,102.4,135/86,True
5,6.5,137.0,87.5,104.0,137/87,True
6,7.0,138.6,89.0,105.5,139/89,True
7,8.0,141.9,92.0,108.6,142/92,True



Оценка АД по СРПВ [P01]:
 pwv_ms  bp_str  map_est  in_range
   8.84  145/95    111.2      True
   6.18  136/87    103.0      True
   3.71  128/79     95.4      True
   3.13  126/77     93.6      True
   4.00  129/80     96.3      True
   5.35  133/84    100.4      True
   5.80  135/85    101.8      True
   6.96  139/89    105.4      True
   5.35  133/84    100.4      True
   3.44  127/78     94.5      True
  11.27 153/102    118.7      True
   4.13  129/80     96.7      True
   4.88  132/83     99.0      True


,signal_idx,pwv_ms,bp_str,map_est,in_range
0,1,8.84,145/95,111.2,True
1,2,6.18,136/87,103.0,True
2,3,3.71,128/79,95.4,True
3,4,3.13,126/77,93.6,True
4,5,4.00,129/80,96.3,True
5,6,5.35,133/84,100.4,True
6,7,5.80,135/85,101.8,True
7,8,6.96,139/89,105.4,True
8,9,5.35,133/84,100.4,True
9,10,3.44,127/78,94.5,True
